Documentation: [readme.md](readme.md). The first code cell switches the process **cwd** to **`deploy/`** so `utils.py` imports work whether you start Jupyter from the **repo root** or from **`deploy/`**.

Environment: from the **repository root**, run **`uv sync`** (creates **`.venv/`** at the repo root) then **`uv run jupyter notebook deploy/deploy.ipynb`** (see readme).

**Firmware `.bin`**: store under **`deploy/bin/`** or rely on auto-download inside **`flash_with_esptool`** / **`run_full_flash`** (see readme).

Deployments are **Step 1–3** below; step code is **uncommented** so you can run each cell as needed. **Run all** will execute Step 1, then 2, then 3 in order (often wrong for one device: Step 3 is usually for **existing** boards only, not right after Step 2 on a new install). Run only the cells that match your task.

## Configuration

Run the **setup** cell below, then **Serial port** to list USB devices and optionally pick one for **`CFG.serial_port`** (default **`/dev/tty.usbmodem101`**).

You can also edit **`CFG`** directly: `circuitpy_root`, and optional **`circuitpython_bin`** / **`circuitpython_board_id`** / **`circuitpython_locale`** / **`circuitpython_download_fallback_url`**`. Settings backups use **`device_id`** from the device `settings.toml` under **`deploy/settings_backups/<device_id>/`**.

In [9]:
import sys
from pathlib import Path

# So ``import notebook_env`` works from repo root or from deploy/
_repo = Path.cwd().resolve()
_deploy_candidate = _repo if (_repo / "utils.py").is_file() else _repo / "deploy"
if str(_deploy_candidate) not in sys.path:
    sys.path.insert(0, str(_deploy_candidate))

import notebook_env

notebook_env.activate()

from utils import (
    DeployConfig,
    copy_firmware_to_circuitpy,
    flash_with_esptool,
    list_serial_ports,
    pick_serial_port_interactive,
    run_full_flash,
    run_update_only,
)

# Override defaults, e.g. DeployConfig(circuitpy_root=Path("/media/youruser/CIRCUITPY"))  # Linux CIRCUITPY path
CFG = DeployConfig()
CFG

DeployConfig(serial_port='/dev/tty.usbmodem101', circuitpy_root=PosixPath('/Volumes/CIRCUITPY'), circuitpython_bin=PosixPath('/Users/silvioheinze/coding/firmware/deploy/bin/adafruit-circuitpython-espressif_esp32s3_devkitc_1_n8r8-de_DE.bin'), circuitpython_board_id='espressif_esp32s3_devkitc_1_n8r8', circuitpython_locale='de_DE', circuitpython_download_fallback_url='https://downloads.circuitpython.org/bin/espressif_esp32s3_devkitc_1_n8r8/en_GB/adafruit-circuitpython-espressif_esp32s3_devkitc_1_n8r8-en_GB-10.1.4.bin', do_erase_flash=True, do_write_firmware=True, wait_for_circuitpy_mount=True, wait_timeout_s=120.0, poll_interval_s=1.0, do_copy_firmware_tree=True, full_flash_settings=PosixPath('/Users/silvioheinze/coding/firmware/deploy/settings.toml'))

### Serial port

Run the next cell to **probe USB serial ports** (requires `pyserial`). It **bootstraps `deploy/`** like the setup cell, so it still works after a **kernel restart** without re-running setup. Type a **number** and Enter to set **`CFG.serial_port`**, or **Enter alone** to keep the current value (default **`/dev/tty.usbmodem101`**). If **`CFG`** does not exist yet, the cell creates it.

In [10]:
import sys
from pathlib import Path

_repo = Path.cwd().resolve()
_deploy_candidate = _repo if (_repo / "utils.py").is_file() else _repo / "deploy"
if str(_deploy_candidate) not in sys.path:
    sys.path.insert(0, str(_deploy_candidate))

import notebook_env

notebook_env.activate()

from utils import DeployConfig, pick_serial_port_interactive

if "CFG" not in globals():
    CFG = DeployConfig()

pick_serial_port_interactive(CFG)
CFG

Available serial ports:
  1) /dev/cu.debug-console — 'n/a'
  2) /dev/cu.Bluetooth-Incoming-Port — 'n/a'
  3) /dev/cu.BoseQuietControl30 — 'n/a'
  4) /dev/cu.usbmodem8DB3AD34BD0B1 — 'ESP32-S3-DevKitC-1-N8R8'
Using serial_port: /dev/cu.usbmodem8DB3AD34BD0B1


DeployConfig(serial_port='/dev/cu.usbmodem8DB3AD34BD0B1', circuitpy_root=PosixPath('/Volumes/CIRCUITPY'), circuitpython_bin=PosixPath('/Users/silvioheinze/coding/firmware/deploy/bin/adafruit-circuitpython-espressif_esp32s3_devkitc_1_n8r8-de_DE.bin'), circuitpython_board_id='espressif_esp32s3_devkitc_1_n8r8', circuitpython_locale='de_DE', circuitpython_download_fallback_url='https://downloads.circuitpython.org/bin/espressif_esp32s3_devkitc_1_n8r8/en_GB/adafruit-circuitpython-espressif_esp32s3_devkitc_1_n8r8-en_GB-10.1.4.bin', do_erase_flash=True, do_write_firmware=True, wait_for_circuitpy_mount=True, wait_timeout_s=120.0, poll_interval_s=1.0, do_copy_firmware_tree=True, full_flash_settings=PosixPath('/Users/silvioheinze/coding/firmware/deploy/settings.toml'))

## Step 1 — Flash the board

Serial flashing only (**`erase_flash`** if enabled, then **`write_flash`**). Resolves a **`.bin`** via **`ensure_circuitpython_bin`** (existing file, `deploy/bin/`, download index, or fallback). Does **not** copy the repo onto **`CIRCUITPY`**.

Use **BOOT + RESET** on ESP32-S3 if the port does not appear. After this step, reset the board and wait until **`CIRCUITPY`** mounts before Step 2.

In [ ]:
flash_with_esptool(CFG)

## Step 2 — Copy firmware to a new board

For a **new** board (or right after a full flash) when you want the repo **`firmware/`** tree plus **`deploy/settings.toml`** on **`CIRCUITPY`**. Waits for **`CFG.circuitpy_root`**, then copies.

While the **`firmware/`** tree is copied, the cell output shows a **tqdm progress bar** (per file, with the current path in the postfix). Install deps with **`uv sync`** at the repo root if the bar does not appear.

Do **not** run Step 3 on the same device in the same session unless you intend the backup/slot workflow (Step 3 overwrites the tree then restores slot settings).

In [12]:
copy_firmware_to_circuitpy(CFG)

Found: /Volumes/CIRCUITPY
Firmware copy: 208 files -> /Volumes/CIRCUITPY


Firmware copy:   0%|          | 0/208 [00:00<?, ?file/s]

Copied firmware tree /Users/silvioheinze/coding/firmware/firmware -> /Volumes/CIRCUITPY (208 files)
Copied /Users/silvioheinze/coding/firmware/deploy/settings.toml -> /Volumes/CIRCUITPY/settings.toml


## Step 3 — Update firmware (existing board)

For a board **already in use**: refresh the **`firmware/`** tree from the repo while keeping WiFi and other settings; backups go under **`deploy/settings_backups/<device_id>/`** (from `device_id` in `settings.toml`). **`CIRCUITPY`** must already be mounted. **No** esptool.

The firmware tree copy uses the same **tqdm progress bar** as Step 2. After the copy, any **new keys** present in the repo’s **`firmware/settings.toml`** and **`firmware/startup.toml`** on **`CIRCUITPY`** but missing from your backed-up copies (when those backups exist) are merged into the slot backup the same way; then the merged files are written back to the board.

Skip Steps 1–2 when you only need an application update.

In [ ]:
run_update_only(CFG)

## Optional — List serial ports

Requires **`pyserial`**. Use the printed device path for **`CFG.serial_port`**.

In [ ]:
list_serial_ports()

## Optional — Shortcut for a new board

**`run_full_flash(CFG)`** runs **Step 1** then **Step 2** in one call (not Step 3). Use this **instead** of running the Step 1 and Step 2 cells above; leave the next line commented if you already use those cells, or you will flash twice.

In [ ]:
# run_full_flash(CFG)  # uncomment only if you skip Step 1 and Step 2 cells above